# Aula 6 — Coleta de Dados da B3: do ticker ao Parquet

## Estudo de caso: montando a base de dados de um pipeline quantitativo

Nas aulas anteriores os dados **já vinham prontos**: `fetch_openml`, um CSV limpo no repositório.
Isso é uma conveniência didática — e uma mentira sobre como o trabalho real acontece.

Em um projeto de verdade, **a coleta é a primeira fase do pipeline** e costuma consumir mais tempo
do que a modelagem. É nela que você decide:

- **Quais** ativos entram na base (o *universo investível*);
- **Qual** preço é o preço certo (spoiler: `Close` quase nunca é);
- **Como** detectar que a fonte te entregou lixo;
- **Onde** guardar os dados para que a próxima execução não baixe tudo de novo.

### O que vamos construir

Um coletor completo, em 8 fases, que replica a arquitetura de um pipeline quant de produção:

| Fase | O que fazemos |
|------|---------------|
| **0** | Preparar o ambiente e as bibliotecas |
| **1** | Mapear as fontes de dados da B3 (e o que cada uma cobra de você) |
| **2** | Descobrir o **universo investível** — a lista de papéis negociados |
| **3** | Coletar as **cotações OHLCV** via `yfinance` |
| **4** | Coletar os **indicadores macro** no BCB (SGS) |
| **5** | **Auditar a qualidade** do que foi coletado |
| **6** | **Armazenar** em Parquet (e medir por que não CSV) |
| **7** | Implementar a **coleta incremental** |
| **8** | Um mini-EDA para provar que a base serve para modelar |

### Pré-requisito conceitual

Nenhum. Mas é útil saber que um **ticker** da B3 tem a forma `PETR4`, `VALE3`, `KNRI11`, e que no
Yahoo Finance ele ganha o sufixo `.SA` (de *São Paulo*): `PETR4.SA`.

---

---

# FASE 0 — PREPARAÇÃO DO AMBIENTE

Vamos usar quatro bibliotecas externas:

| Biblioteca | Para quê |
|------------|----------|
| `yfinance` | baixar cotações históricas do Yahoo Finance |
| `python-bcb` | acessar o sistema SGS do Banco Central (IPCA, Selic...) |
| `lxml` | fazer o *parsing* das tabelas HTML do Fundamentus |
| `pyarrow` | ler e escrever arquivos Parquet |

A célula abaixo instala o que estiver faltando. Se você já rodou uma vez, ela é praticamente
instantânea — o `pip` apenas confirma que está tudo lá.

In [ ]:
# Rode UMA vez. Se já estiver instalado, o pip apenas confirma e sai.
%pip install -q yfinance python-bcb lxml pyarrow

In [ ]:
# Bibliotecas usadas na aula inteira
import time
import warnings
from datetime import date, timedelta
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yfinance as yf

warnings.filterwarnings("ignore")          # silencia avisos de depreciacao do yfinance
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (12, 4)

# Onde os dados brutos vao morar. 'raw' = como vieram da fonte, sem tratamento.
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("yfinance :", yf.__version__)
print("pandas   :", pd.__version__)
print("dados em :", DATA_DIR.resolve())

### Por que uma pasta `data/raw`?

A convenção `raw` / `processed` separa dois mundos:

- **`raw`** — o dado **exatamente como a fonte entregou**. Nunca é editado à mão. Se você
  desconfiar de um resultado, é aqui que você volta para conferir.
- **`processed`** — o resultado das suas transformações. É **descartável**: sempre pode ser
  regerado a partir do `raw` rodando o código de novo.

> ⚠️ **Nunca versione a pasta `data/` no Git.** Parquets de cotações chegam a centenas de MB, mudam
> todo dia e inflam o repositório para sempre (o Git guarda todas as versões). Coloque `data/` no
> `.gitignore` e versione **o código que baixa os dados** — é ele que é reprodutível.

---

---

# FASE 1 — DE ONDE VÊM OS DADOS DA B3?

Antes de escrever a primeira linha de coleta, é preciso saber **o que existe**. Para o mercado
brasileiro, estas são as fontes realistas:

| Fonte | O que fornece | Custo | Pegadinhas |
|-------|---------------|-------|------------|
| **Yahoo Finance** (`yfinance`) | OHLCV diário/intradiário de ações, FIIs, índices, câmbio, commodities | grátis | API não-oficial; pode mudar sem aviso; *rate limit* não documentado |
| **Fundamentus** (scraping) | lista de todos os papéis + indicadores fundamentalistas | grátis | HTML pode mudar; exige `User-Agent`; uso pesado é abusivo |
| **BCB / SGS** | séries macro (IPCA, IGP-M, INCC, Selic, câmbio PTAX) | grátis | API **oficial** e estável; limite de janela para séries diárias |
| **B3 — arquivo COTAHIST** | histórico oficial de negócios, layout posicional | grátis | TXT de largura fixa, um por ano, dezenas de MB |
| **Provedores pagos** (Economatica, Quantum, Refinitiv) | tudo, já tratado e auditado | caro | — |

### A escolha desta aula

Usamos **Yahoo + Fundamentus + BCB**. Não porque sejam as melhores, mas porque são as que permitem
a você rodar este notebook agora, sem cadastro e sem cartão de crédito.

### ⚠️ Sobre a diferença entre "grátis" e "confiável"

O Yahoo Finance é uma fonte **não-oficial**. Ele já entregou séries com *splits* aplicados errado,
volumes zerados em pregão normal e tickers que somem e voltam.

Isso **não** inviabiliza o uso didático nem exploratório. Mas define uma regra que vale para
qualquer projeto de dados:

> **Toda fonte externa é suspeita até você auditá-la.** É exatamente o que faremos na Fase 5.

---

---

# FASE 2 — O UNIVERSO INVESTÍVEL

## §2.1 — O problema do universo

A pergunta "quais ações eu analiso?" parece trivial e não é. Se você digitar à mão uma lista de 10
papéis que **você conhece**, acabou de introduzir o erro mais caro da análise quantitativa:

> 🎯 **Viés de sobrevivência** (*survivorship bias*): você só lembra das empresas que deram certo.
> As que quebraram, fecharam capital ou foram incorporadas sumiram da sua memória — e do seu
> backtest. Resultado: a estratégia parece lucrativa porque você, sem perceber, já sabia o futuro.

A defesa é coletar o universo **de uma fonte**, não da memória. O Fundamentus publica a lista
completa de papéis negociados em duas páginas HTML.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}


def listar_papeis_fundamentus(url: str, rotulo: str) -> list:
    """Baixa uma tabela do Fundamentus e devolve os tickers com sufixo '.SA'.

    O Fundamentus responde com uma pagina HTML contendo UMA tabela grande.
    pd.read_html converte toda tag <table> do HTML em DataFrame -- pegamos a primeira.
    """
    try:
        resposta = requests.get(url, headers=HEADERS, timeout=15)
        resposta.raise_for_status()
        tabelas = pd.read_html(StringIO(resposta.text))   # StringIO evita aviso de depreciacao
        if not tabelas:
            print(f"[{rotulo}] nenhuma tabela encontrada na pagina.")
            return []
        tickers = tabelas[0]["Papel"].astype(str).str.strip().tolist()
        print(f"[{rotulo}] {len(tickers)} papeis encontrados.")
        return [f"{t}.SA" for t in tickers]
    except Exception as e:
        print(f"[{rotulo}] falhou: {type(e).__name__} -- {e}")
        return []


acoes = listar_papeis_fundamentus("https://fundamentus.com.br/resultado.php", "Acoes")
fiis = listar_papeis_fundamentus("https://fundamentus.com.br/fii_resultado.php", "FIIs")

print()
print("Amostra de acoes:", acoes[:8])
print("Amostra de FIIs :", fiis[:8])

### 🔎 Três detalhes dessa função que não são decorativos

**1. O `User-Agent`.** Sem esse cabeçalho, a requisição se identifica como `python-requests/2.x`.
Muitos servidores bloqueiam esse valor por padrão. Passar um `User-Agent` de navegador é prática
corrente de scraping — mas note que você está **se declarando outra coisa**. É aceitável para uso
pessoal e leve; fazer milhares de requisições assim não é.

**2. O `timeout=15`.** Uma requisição sem `timeout` pode **travar para sempre** se o servidor
aceitar a conexão e nunca responder. Em um notebook você mata o kernel; em um job agendado às 3h da
manhã, ele simplesmente nunca termina. **Sempre** ponha timeout.

**3. O `try/except` que devolve lista vazia.** A coleta **não derruba o notebook** quando a fonte
cai — ela avisa e segue. Esse padrão (*falhar suave e continuar*) se repete em todas as funções
desta aula, porque numa coleta de centenas de tickers é certo que alguns vão falhar.

---
## §2.2 — Recortando um universo de trabalho

Baixar o histórico completo de ~1.500 papéis leva vários minutos e é desnecessário para aprender.
Vamos trabalhar com um recorte — mas **explícito**, para que você saiba qual viés está aceitando.

In [ ]:
# Blue chips: acoes muito liquidas da B3
BLUE_CHIPS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "WEGE3.SA", "BBAS3.SA"]

# FIIs de tijolo e de logistica, bastante negociados
FIIS = ["KNRI11.SA", "HGLG11.SA", "XPLG11.SA"]

# Indices de referencia (benchmarks)
# IFIX.SA fica aqui DE PROPOSITO: ele existe no Yahoo, mas vem quase vazio.
# Na Fase 5 vamos flagra-lo -- e essa e a graca. XFIX11 e o ETF que replica o IFIX.
INDICES = ["^BVSP", "IFIX.SA", "XFIX11.SA"]

# Cambio e commodities: fatores externos que movem a bolsa brasileira
FATORES = ["BRL=X", "GC=F", "CL=F"]     # Dolar/Real, Ouro, Petroleo WTI

UNIVERSO = BLUE_CHIPS + FIIS + INDICES + FATORES
print(f"{len(UNIVERSO)} ativos no universo de trabalho:")
print(UNIVERSO)

### 📌 Note a anatomia dos tickers

| Padrão | Significado | Exemplo |
|--------|-------------|---------|
| `XXXX3` | ação **ordinária** (com voto) | `VALE3` |
| `XXXX4` | ação **preferencial** (sem voto, prioridade em dividendos) | `PETR4` |
| `XXXX11` | **FII**, ETF ou *unit* | `KNRI11` |
| `.SA` | sufixo do Yahoo para a B3 | `PETR4.SA` |
| `^` | prefixo do Yahoo para **índices** | `^BVSP` |
| `XXXX11` (ETF) | fundo de índice negociado em bolsa | `XFIX11` |
| `=X` | par de **câmbio** | `BRL=X` |
| `=F` | contrato **futuro** (commodity) | `CL=F` |

O Ibovespa é `^BVSP` (sem `.SA`) e o IFIX é `IFIX.SA`. Essa inconsistência não tem lógica — é como
o Yahoo catalogou. **Convenção de fonte não se discute, se documenta.**

---

---

# FASE 3 — COLETANDO AS COTAÇÕES (OHLCV)

## §3.1 — O que o `yfinance` devolve

Uma linha por **(ativo, pregão)**, com estas colunas:

| Coluna | Significado |
|--------|-------------|
| `Open` | preço do **primeiro** negócio do dia |
| `High` | **maior** preço negociado no dia |
| `Low` | **menor** preço negociado no dia |
| `Close` | preço do **último** negócio (preço de fechamento) |
| `Adj Close` | fechamento **ajustado** por proventos e desdobramentos |
| `Volume` | quantidade de **ações** negociadas (não é dinheiro) |

Esse conjunto é chamado de **OHLCV**. É o formato universal de dados de mercado — de ações a
criptomoedas, do candle de 1 minuto ao mensal.

### ⚠️ O detalhe que separa análise correta de análise errada: `auto_adjust`

A partir da versão 0.2.51 o `yfinance` passou a usar `auto_adjust=True` por padrão, o que
**sobrescreve** `Close` com o valor ajustado e **apaga** a coluna `Adj Close`. Nós passamos
`auto_adjust=False` de propósito: queremos as **duas** colunas, porque cada uma responde a uma
pergunta diferente — e vamos comparar as duas daqui a pouco.

In [ ]:
def coletar_cotacoes(tickers, start=None, periodo="max") -> pd.DataFrame:
    """Baixa OHLCV do Yahoo e devolve um DataFrame LONGO (uma linha por ativo/pregao).

    Args:
        tickers: lista de tickers (ou uma string com um unico ticker).
        start:   data inicial 'YYYY-MM-DD'. Se None, usa `periodo`.
        periodo: janela quando start e None ('max', '5y', '1y', '6mo'...).

    Returns:
        DataFrame com colunas Date, Ticker, Open, High, Low, Close, Adj Close, Volume.
        DataFrame vazio se a coleta falhar (a funcao nunca levanta excecao).
    """
    if isinstance(tickers, str):
        tickers = [tickers]

    try:
        bruto = yf.download(
            tickers,
            period=None if start else periodo,
            start=start,
            group_by="ticker",     # organiza as colunas por ticker
            auto_adjust=False,     # PRESERVA a coluna 'Adj Close' -- ver texto acima
            progress=False,
            threads=True,          # baixa varios tickers em paralelo
        )
    except Exception as e:
        print(f"Falha no download: {type(e).__name__} -- {e}")
        return pd.DataFrame()

    if bruto is None or bruto.empty:
        print("Yahoo devolveu tabela vazia.")
        return pd.DataFrame()

    # O yfinance devolve um DataFrame LARGO com colunas em dois niveis:
    #   (PETR4.SA, Close), (PETR4.SA, Volume), (VALE3.SA, Close), ...
    # Precisamos do formato LONGO, que e o que empilha bem e salva bem em Parquet.
    partes = []
    if isinstance(bruto.columns, pd.MultiIndex):
        disponiveis = set(bruto.columns.get_level_values(0))
        for tk in tickers:
            if tk not in disponiveis:
                print(f"  [aviso] sem dados para {tk}")
                continue
            parte = bruto[tk].copy()
            parte["Ticker"] = tk
            partes.append(parte.reset_index())
    else:                                   # caso de um unico ticker
        parte = bruto.copy()
        parte["Ticker"] = tickers[0]
        partes.append(parte.reset_index())

    if not partes:
        return pd.DataFrame()

    df = pd.concat(partes, ignore_index=True)
    df = df.rename(columns={"Datetime": "Date", "index": "Date"})
    df = df.dropna(subset=["Close"])         # dia sem fechamento = dia sem pregao para o ativo

    ordem = ["Date", "Ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    df = df[[c for c in ordem if c in df.columns]]
    return df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

In [ ]:
# Coleta de verdade: ~10 anos de historico para os 14 ativos do universo.
# Leva de 10 a 40 segundos, dependendo da sua conexao e do humor do Yahoo.
t0 = time.time()
cotacoes = coletar_cotacoes(UNIVERSO, periodo="10y")
print(f"\nColetado em {time.time() - t0:.1f}s")

print("Formato :", cotacoes.shape)
print("Periodo :", cotacoes['Date'].min().date(), "ate", cotacoes['Date'].max().date())
print("Ativos  :", cotacoes['Ticker'].nunique())
cotacoes.head()

In [ ]:
# Quantos pregoes vieram por ativo? (a primeira checagem que todo mundo deveria fazer)
resumo = (
    cotacoes.groupby("Ticker")
    .agg(
        pregoes=("Date", "count"),
        inicio=("Date", "min"),
        fim=("Date", "max"),
        preco_medio=("Close", "mean"),
    )
    .sort_values("pregoes")
)
resumo["inicio"] = resumo["inicio"].dt.date
resumo["fim"] = resumo["fim"].dt.date
resumo.round(2)

### 🔎 Como ler essa tabela de resumo

Três perguntas que ela responde de imediato:

1. **Todos os ativos têm um número parecido de pregões?** Se um tem metade dos outros, ou o papel é
   recente, ou a série tem buracos. Nos dois casos, usá-lo em uma matriz de correlação vai gerar
   resultado enviesado.
2. **As datas de início batem?** Commodities e câmbio (`GC=F`, `BRL=X`) negociam em feriados
   brasileiros; ações não. Os calendários **não coincidem** — e isso vai importar na hora de juntar
   as séries.
3. **O preço médio faz sentido?** `^BVSP` na casa das dezenas de milhares (pontos do índice),
   ações na casa das dezenas de reais. Um ativo com preço médio de 0,001 é um alerta de erro de
   unidade.

---
## §3.2 — `Close` ou `Adj Close`? A pergunta que decide o resultado

Imagine uma ação que custa R$ 100 e paga R$ 5 de dividendo. No dia seguinte ela abre valendo
R$ 95 — não porque a empresa piorou, mas porque **o dinheiro saiu do caixa e foi para o seu
bolso**. É o *ex-dividend date*.

- **`Close`** registra a queda: R$ 100 → R$ 95. Foi isso que apareceu no home broker.
- **`Adj Close`** reescreve o passado para que a série **não tenha esse degrau artificial**: é o
  preço que corresponde a um investidor que **reinvestiu** cada provento recebido.

O mesmo vale para *splits*: se uma ação de R$ 100 é desdobrada 1:2, o `Close` cai para R$ 50 da
noite para o dia. Ninguém perdeu metade do patrimônio.

> 🎯 **A regra**: para **calcular retorno**, use `Adj Close`. Para **simular uma ordem** (quanto eu
> pagaria hoje, quantas cotas cabem no meu dinheiro), use `Close`.

Vamos medir o tamanho do estrago.

In [ ]:
def retorno_acumulado(serie: pd.Series) -> float:
    """Retorno percentual entre o primeiro e o ultimo ponto da serie."""
    return (serie.iloc[-1] / serie.iloc[0] - 1) * 100


linhas = []
for tk in BLUE_CHIPS + FIIS:
    d = cotacoes[cotacoes["Ticker"] == tk].sort_values("Date")
    if d.empty or "Adj Close" not in d.columns:
        continue
    linhas.append({
        "Ticker": tk,
        "ret_Close_%": retorno_acumulado(d["Close"]),
        "ret_AdjClose_%": retorno_acumulado(d["Adj Close"]),
    })

comparativo = pd.DataFrame(linhas)
comparativo["diferenca_pp"] = comparativo["ret_AdjClose_%"] - comparativo["ret_Close_%"]
comparativo.sort_values("diferenca_pp", ascending=False).round(1)

### 📌 Achado: a diferença é de **dezenas de pontos percentuais**

Olhe a coluna `diferenca_pp` (diferença em pontos percentuais). Em papéis que pagam muito dividendo
— bancos e FIIs, tipicamente — o retorno medido pelo `Close` **subestima** o ganho real de forma
brutal, porque joga fora todo o dinheiro que foi pago ao cotista.

Nos FIIs o efeito é ainda mais dramático: como eles são obrigados por lei a distribuir no mínimo
95% do lucro semestral, quase todo o retorno chega via provento. Um FII pode ter cotação
**praticamente parada** há anos e ainda assim ter dado um retorno excelente.

> ⚠️ **Este é o erro nº 1 de quem começa em finanças quantitativas.** Um backtest de estratégia de
> dividendos construído sobre `Close` conclui que dividendos destroem valor — o que é o oposto da
> realidade. O modelo estava certo; o dado é que estava errado.

Vamos ver isso graficamente.

In [ ]:
alvo = "ITUB4.SA" if (cotacoes["Ticker"] == "ITUB4.SA").any() else cotacoes["Ticker"].iloc[0]
d = cotacoes[cotacoes["Ticker"] == alvo].sort_values("Date")

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(d["Date"], d["Close"] / d["Close"].iloc[0] * 100,
        label="Close (o que aparece no home broker)", lw=1.2)
ax.plot(d["Date"], d["Adj Close"] / d["Adj Close"].iloc[0] * 100,
        label="Adj Close (com proventos reinvestidos)", lw=1.2)
ax.set(title=f"{alvo} — mesmo ativo, duas historias (base 100)",
       xlabel="Data", ylabel="Indice (inicio = 100)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 🔎 Como ler este gráfico

As duas linhas partem **do mesmo ponto** (base 100) e vão se afastando. O afastamento **não é
ruído**: é exatamente o dinheiro que foi distribuído ao acionista ao longo do período.

A distância entre as linhas cresce em degraus — cada degrau é uma data-ex. E ela **só cresce**,
nunca encolhe, porque provento pago não volta.

> 📌 Se você olhasse apenas a linha `Close`, concluiria que o ativo andou de lado. A linha
> `Adj Close` mostra que quem ficou posicionado ganhou bem mais do que a cotação sugere.

---

---

# FASE 4 — DADOS MACROECONÔMICOS (BCB / SGS)

## §4.1 — Por que sair da bolsa para entender a bolsa

Preço de ação não se move no vácuo. Quando a Selic sobe:

- a renda fixa fica mais atraente → dinheiro sai da bolsa;
- o crédito encarece → empresas alavancadas sofrem;
- o custo de oportunidade sobe → o *yield* exigido dos FIIs sobe → a cota cai.

Essas variáveis são **features exógenas**: não vêm do histórico do próprio ativo, mas explicam
parte do que acontece com ele. Em modelos de previsão, elas costumam adicionar informação que os
preços passados sozinhos não têm.

O **SGS** (Sistema Gerenciador de Séries Temporais) do Banco Central publica milhares delas, com
uma API **oficial, gratuita e estável** — o oposto do Yahoo.

| Série | Código SGS | O que mede | Frequência |
|-------|-----------|------------|------------|
| IPCA | 433 | inflação oficial ao consumidor | mensal |
| IGP-M | 189 | índice geral de preços (reajuste de aluguéis) | mensal |
| INCC | 192 | custo da construção civil | mensal |
| Selic | 11 | taxa básica de juros, ao dia | **diária** |

> 💡 Quer outra série? O catálogo completo está em
> <https://www3.bcb.gov.br/sgspub/> — busque pelo nome e anote o código.

In [ ]:
SGS_CODIGOS = {"ipca": 433, "igpm": 189, "incc": 192, "selic": 11}

try:
    from bcb import sgs
    TEM_PYTHON_BCB = True
except ImportError:
    TEM_PYTHON_BCB = False
    print("python-bcb nao instalado -- usaremos a API REST direto.")


def coletar_sgs_rest(nome: str, codigo: int, inicio: date, fim: date) -> pd.DataFrame:
    """Plano B: fala com a API REST do BCB usando apenas requests.

    Endpoint publico e documentado:
      https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json
    Devolve JSON no formato [{"data": "01/01/2020", "valor": "0.21"}, ...].
    """
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    params = {
        "formato": "json",
        "dataInicial": inicio.strftime("%d/%m/%Y"),
        "dataFinal": fim.strftime("%d/%m/%Y"),
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    bruto = pd.DataFrame(r.json())
    if bruto.empty:
        return pd.DataFrame(columns=["Date", "Indicador", "Valor"])
    return pd.DataFrame({
        "Date": pd.to_datetime(bruto["data"], format="%d/%m/%Y"),
        "Indicador": nome,
        # o BCB devolve numero como texto com PONTO decimal, mas ja veio com virgula
        # em versoes antigas da API -- por isso normalizamos antes de converter
        "Valor": pd.to_numeric(bruto["valor"].astype(str).str.replace(",", ".", regex=False),
                               errors="coerce"),
    }).dropna(subset=["Valor"])


def coletar_macro(codigos: dict, anos: int = 10) -> pd.DataFrame:
    """Coleta as series do SGS UMA A UMA, tolerando falha individual.

    Formato LONGO (Date, Indicador, Valor): series com frequencias diferentes
    (mensal x diaria) nao cabem no mesmo formato largo sem gerar um mar de nulos.
    """
    fim = date.today()
    # A API do BCB recusa janelas > 10 anos para series DIARIAS (caso da Selic).
    # Pedimos 10 anos menos um mes de folga.
    inicio = fim - timedelta(days=365 * anos - 30)

    partes = []
    for nome, codigo in codigos.items():
        try:
            if TEM_PYTHON_BCB:
                bruto = sgs.get({nome: codigo}, start=inicio.strftime("%Y-%m-%d"))
                bruto.index.name = "Date"
                parte = (bruto.reset_index()
                              .melt(id_vars="Date", var_name="Indicador", value_name="Valor"))
            else:
                parte = coletar_sgs_rest(nome, codigo, inicio, fim)
            parte = parte.dropna(subset=["Valor"])
            partes.append(parte)
            print(f"  ok  {nome:6s} (SGS {codigo:4d}): {len(parte):5d} pontos")
        except Exception as e:
            print(f"  ERRO {nome:6s} (SGS {codigo:4d}): {type(e).__name__} -- {e}")

    if not partes:
        return pd.DataFrame(columns=["Date", "Indicador", "Valor"])
    return (pd.concat(partes, ignore_index=True)
              .sort_values(["Indicador", "Date"])
              .reset_index(drop=True))


print("Coletando series do BCB...")
macro = coletar_macro(SGS_CODIGOS)
print("\nFormato:", macro.shape)
macro.head()

### 🔎 Duas decisões de projeto escondidas nesse código

**1. Uma série por vez, cada uma no seu `try`.** Seria mais curto pedir as quatro de uma vez. Mas
aí, se o INCC estiver indisponível naquele minuto, você perde **as quatro**. Coletar em laço com
tratamento individual é mais lento e infinitamente mais robusto — a mesma lógica do `try/except` da
Fase 2.

**2. Formato longo `(Date, Indicador, Valor)`.** Por que não uma coluna por indicador? Porque o
IPCA é **mensal** e a Selic é **diária**. No formato largo, 95% da coluna `ipca` seria `NaN`. O
formato longo aceita frequências diferentes sem desperdício — e é trivial pivotar para largo quando
você precisar (`df.pivot(...)`), como faremos na Fase 8.

> 💡 Esse formato tem nome: *tidy data*, ou formato **tidy/long**. É o mesmo princípio de uma tabela
> normalizada em banco de dados.

In [ ]:
# Cada indicador tem sua propria frequencia e sua propria unidade -- confira:
(macro.groupby("Indicador")
      .agg(pontos=("Date", "count"),
           inicio=("Date", "min"),
           fim=("Date", "max"),
           valor_min=("Valor", "min"),
           valor_max=("Valor", "max"))
      .assign(inicio=lambda d: d["inicio"].dt.date, fim=lambda d: d["fim"].dt.date)
      .round(3))

### ⚠️ Atenção à **unidade** de cada série

Repare nos valores mínimo e máximo:

- **IPCA, IGP-M, INCC** vêm em **variação percentual mensal** (ex.: `0.52` = +0,52% no mês). Não
  são o nível do índice — para ter o acumulado do ano, você precisa **compor**:
  $\prod (1 + r_i) - 1$, nunca somar.
- **Selic (código 11)** vem como **taxa diária** em percentual (ex.: `0.047` ≈ 0,047% ao dia). Para
  a taxa anualizada equivalente: $(1 + r_d)^{252} - 1$.

> 🎯 Somar variações percentuais é o erro silencioso mais comum em análise macro. Para valores
> pequenos o resultado até se aproxima do certo, o que é justamente o que torna o erro difícil de
> pegar. **Percentual se compõe, não se soma.**

---

---

# FASE 5 — AUDITORIA DE QUALIDADE

Os dados chegaram. A tentação é começar a modelar. **Não comece.**

Um modelo treinado sobre dado sujo produz um número, e o número parece respeitável — gráficos
bonitos, métricas com quatro casas decimais. É o pior tipo de erro: **silencioso**.

Vamos rodar cinco checagens. Elas cabem em meia página de código e valem por semanas de depuração.

## §5.1 — As cinco perguntas

In [ ]:
# CHECAGEM 1 -- Valores ausentes por coluna
print("=== 1. Valores ausentes ===")
nulos = cotacoes.isna().sum()
print(nulos[nulos > 0] if nulos.sum() else "Nenhum valor ausente.")

# CHECAGEM 2 -- Duplicatas na chave primaria (Date, Ticker)
print("\n=== 2. Duplicatas em (Date, Ticker) ===")
dups = cotacoes.duplicated(subset=["Date", "Ticker"]).sum()
print(f"{dups} linhas duplicadas")
print("A chave (Date, Ticker) deve ser UNICA: um ativo tem um unico fechamento por pregao.")

In [ ]:
# CHECAGEM 3 -- Pregoes com volume zero
print("=== 3. Volume zero ===")
if "Volume" in cotacoes.columns:
    vol_zero = (
        cotacoes[cotacoes["Volume"] == 0]
        .groupby("Ticker").size().sort_values(ascending=False)
    )
    print(vol_zero.head(10) if len(vol_zero) else "Nenhum pregao com volume zero.")
    print(f"\nTotal: {int((cotacoes['Volume'] == 0).sum())} linhas "
          f"({(cotacoes['Volume'] == 0).mean() * 100:.2f}% da base)")

In [ ]:
# CHECAGEM 4 -- Coerencia interna do candle: High >= Open/Close >= Low
print("=== 4. Coerencia OHLC ===")
cols_ohlc = {"Open", "High", "Low", "Close"}
if cols_ohlc.issubset(cotacoes.columns):
    incoerente = cotacoes[
        (cotacoes["High"] < cotacoes[["Open", "Close"]].max(axis=1))
        | (cotacoes["Low"] > cotacoes[["Open", "Close"]].min(axis=1))
    ]
    print(f"{len(incoerente)} candles impossiveis "
          f"(maxima abaixo do fechamento, ou minima acima da abertura)")
    if len(incoerente):
        display(incoerente.head())

In [ ]:
# CHECAGEM 5 -- Saltos de preco absurdos (candidatos a split nao ajustado)
print("=== 5. Variacoes diarias extremas ===")
cot = cotacoes.sort_values(["Ticker", "Date"]).copy()
cot["ret_diario"] = cot.groupby("Ticker")["Close"].pct_change()

LIMIAR = 0.25   # 25% num unico pregao
extremos = cot[cot["ret_diario"].abs() > LIMIAR]

print(f"{len(extremos)} pregoes com |variacao| > {LIMIAR:.0%}\n")
if len(extremos):
    print(extremos.groupby("Ticker")["ret_diario"].agg(["count", "min", "max"]).round(3))
    print("\nOs 5 maiores saltos:")
    print(extremos.reindex(extremos["ret_diario"].abs().sort_values(ascending=False).index)
                  .head(5)[["Date", "Ticker", "Close", "ret_diario"]]
                  .to_string(index=False))

### 📋 Como interpretar os achados

| Achado | Diagnóstico provável | O que fazer |
|--------|---------------------|-------------|
| `Adj Close` nulo em alguns tickers | índices (`^BVSP`) e futuros não têm ajuste — não pagam provento | esperado; use `Close` para esses |
| Volume zero | feriado parcial, ativo ilíquido, ou falha da fonte | investigar; volume zero **crônico** = excluir o ativo |
| Candle incoerente | erro da fonte, sem exceção | descartar a linha |
| Variação > 25% | **split não ajustado**, crash real, ou lixo | ver o caso concreto |

### 🕵️ Cuidado com a conclusão apressada sobre as variações extremas

O reflexo é apagar toda linha com `|ret| > 25%`. **Não faça isso.** A mesma variação pode significar
coisas opostas:

- **Março de 2020** (COVID): o Ibovespa caiu mais de 12% em um dia, com *circuit breaker* acionado
  seis vezes no mês. É **dado real** — e é exatamente o cenário que você mais precisa que o modelo
  tenha visto.
- **Um salto de +100% ou −50% isolado**, sem nada acontecendo no mercado, num papel qualquer: quase
  sempre é **desdobramento/grupamento não ajustado**. É lixo.

A diferença entre os dois casos não está no número — está no **contexto**. Confira a data contra o
que aconteceu no mercado antes de apagar qualquer coisa.

### 🛢️ O caso extremo que provavelmente apareceu na sua saída

Se `CL=F` (petróleo WTI) aparecer com variação de cerca de **−306%** em **20/04/2020**, não é bug:
naquele dia o contrato futuro do WTI fechou em **preço negativo** (−US$ 37) pela primeira vez na
história. Faltava espaço de armazenamento e os detentores do contrato **pagaram** para não receber
o barril.

Isso quebra a própria aritmética do retorno percentual: com preço negativo, `pct_change()` deixa de
ter significado econômico, e qualquer modelo que consuma esse número em log-retorno vai produzir
`NaN` ou `-inf`.

> 📌 Lição: preço negativo é uma **premissa implícita** que quase todo código financeiro faz sem
> perceber — e que o mercado já violou. Séries de futuros de commodities exigem tratamento próprio.

> 🎯 **Regra de ouro da limpeza**: descarte apenas o que você consegue **explicar por que é erro**.
> Descartar o que é apenas incômodo é como treinar um modelo de risco escondendo dele as crises.

---
## §5.2 — Cobertura: todo mundo tem os mesmos pregões?

As cinco checagens olharam **linha a linha**. Falta olhar a base **ativo a ativo**: será que todos
têm um histórico comparável? Vamos medir numa janela recente, para que papéis que estrearam há
pouco não distorçam a comparação.

In [ ]:
# Quantos pregoes cada ativo tem nos ultimos 2 anos?
corte = cotacoes["Date"].max() - pd.Timedelta(days=730)
recorte = cotacoes[cotacoes["Date"] >= corte]

cobertura = recorte.groupby("Ticker")["Date"].nunique().sort_values()
print(f"Janela: {corte.date()} a {recorte['Date'].max().date()}\n")
print(cobertura.to_string())

# Uniao x intersecao dos calendarios
calendarios = [set(g["Date"]) for _, g in recorte.groupby("Ticker")]
datas_uniao = recorte["Date"].nunique()
datas_inter = len(set.intersection(*calendarios))

print(f"\nDatas em que ALGUM ativo negociou (uniao)  : {datas_uniao}")
print(f"Datas em que TODOS negociaram (intersecao) : {datas_inter}")
print(f"Perda ao exigir todos presentes            : "
      f"{(1 - datas_inter / datas_uniao) * 100:.1f}% das datas")

### 📌 Dois achados de uma vez só

**Achado 1 — um ativo quebrado.** `IFIX.SA` aparece com **um punhado de pregões** (às vezes um
único). Não é feriado nem baixa liquidez: o Yahoo simplesmente **não tem** a série histórica do
IFIX, mesmo respondendo ao ticker como se tivesse. Foi por isso que o deixei no universo — é o
exemplo perfeito da Fase 1: *toda fonte externa é suspeita até você auditá-la*. Se você não
conferisse, descobriria isso só depois que o modelo devolvesse um resultado sem sentido.

> 💡 A saída prática: usar o **ETF `XFIX11`**, que replica o IFIX e tem série completa. Dado
> indisponível quase sempre tem um substituto — desde que você saiba que precisa procurar.

**Achado 2 — calendários diferentes.** Ativos brasileiros não negociam em 7 de setembro; ouro e
petróleo, sim. A bolsa americana não abre no Thanksgiving; a B3 abre. Por isso a **interseção** dos
calendários é menor que a união.

E repare no efeito devastador do achado 1 sobre o achado 2: como o `IFIX.SA` tem praticamente uma
data, a interseção de **todos** os ativos desaba para quase nada. **Um único ativo quebrado
inviabiliza o alinhamento da base inteira** — motivo de sobra para tratá-lo antes de seguir.

Isso vai explodir na sua cara em três situações:

1. **Matriz de correlação** — se você alinhar por posição em vez de por data, estará correlacionando
   a terça de um ativo com a quarta de outro. O número sai; o sentido, não.
2. **Preenchimento para trás** (*backward fill*) — completar o feriado brasileiro com o preço do
   **dia seguinte** é usar informação do futuro. É ***look-ahead bias***, o vazamento de dados
   clássico em séries temporais.
3. **Modelos multiativos** — que exigem uma matriz retangular sem buracos.

**A solução correta** tem duas partes:

- alinhe sempre **por data** (`pivot`/`join` no índice de datas, nunca por posição);
- preencha **apenas para frente** (`ffill`): o último preço conhecido é informação que você
  **realmente tinha** naquele momento.

---
## §5.3 — Auditar sem agir não serve para nada

Uma auditoria que só imprime tabelas é decoração. Vamos **agir** sobre o achado 1: descartar os
ativos cuja cobertura é irrecuperavelmente baixa — e deixar registrado **quem** saiu e **por quê**.

In [ ]:
# Regra: menos da metade da cobertura mediana = serie inutilizavel.
# Usamos a MEDIANA (nao a media) porque ela nao e puxada para baixo pelo proprio caso quebrado.
limite = 0.5 * cobertura.median()
descartados = cobertura[cobertura < limite].index.tolist()

print(f"Cobertura mediana na janela: {cobertura.median():.0f} pregoes")
print(f"Limite de aceitacao (50%)  : {limite:.0f} pregoes")
print(f"Descartados                : {descartados if descartados else 'nenhum'}\n")

ATIVOS_OK = [t for t in UNIVERSO if t not in descartados]
cotacoes_ok = cotacoes[cotacoes["Ticker"].isin(ATIVOS_OK)].reset_index(drop=True)

print(f"Universo: {len(UNIVERSO)} -> {len(ATIVOS_OK)} ativos")
print(f"Linhas  : {len(cotacoes):,} -> {len(cotacoes_ok):,}\n")

# E a intersecao dos calendarios, quanto ficou depois da limpeza?
rec_ok = cotacoes_ok[cotacoes_ok["Date"] >= corte]
inter_ok = len(set.intersection(*[set(g["Date"]) for _, g in rec_ok.groupby("Ticker")]))
print(f"Datas na intersecao ANTES da limpeza : {datas_inter:4d} de {datas_uniao}")
print(f"Datas na intersecao DEPOIS da limpeza: {inter_ok:4d} de {datas_uniao} "
      f"({inter_ok / datas_uniao * 100:.0f}%)")

### 📌 Por que um **critério** e não uma lista na mão

Escrever `descartados = ["IFIX.SA"]` resolveria hoje e falharia amanhã, quando outro ticker
quebrasse. Um critério (`< 50% da cobertura mediana`) é **código que continua funcionando** quando
o universo muda — e deixa explícito, para quem ler o notebook daqui a seis meses, qual foi a régua.

E veja o que a limpeza fez com a interseção dos calendários: de praticamente **nenhuma** data comum
para a maior parte delas. O que sobra de perda agora é o desencontro legítimo entre B3, câmbio e
commodities — esse não some, e é com ele que a Fase 8 vai lidar (com `ffill`, nunca `bfill`).

Note também que o corte usa a **mediana**, não a média. A média é puxada para baixo justamente
pelos casos quebrados que queremos detectar — o detector ficaria mais tolerante quanto pior fosse a
base. A mediana ignora os extremos.

---

---

# FASE 6 — ARMAZENAMENTO: POR QUE PARQUET E NÃO CSV

CSV é texto puro: legível, universal e **péssimo** para dados de mercado. Cada número vira uma
sequência de caracteres, e todo tipo de coluna se perde na ida e volta — datas viram string, inteiros
viram float, e o arquivo fica enorme.

**Parquet** é um formato **colunar e binário**:

- guarda cada coluna separada e **comprimida** (colunas de preços têm valores parecidos → comprimem
  muito bem);
- **preserva os tipos** (data continua sendo data ao reler);
- permite ler **só as colunas que você pediu**, sem tocar no resto do arquivo.

Em vez de acreditar, vamos **medir**.

In [ ]:
caminho_parquet = DATA_DIR / "b3_cotacoes.parquet"
caminho_csv = DATA_DIR / "b3_cotacoes.csv"

# AQUECIMENTO: a primeira chamada ao pyarrow paga o custo de carregar a biblioteca.
# Sem isso, o cronometro mede importacao de modulo -- nao velocidade de I/O.
# (Esse cuidado vale para QUALQUER benchmark, nao so aqui.)
aquecimento = DATA_DIR / "_aquecimento.parquet"
cotacoes_ok.head(50).to_parquet(aquecimento, index=False)
_ = pd.read_parquet(aquecimento)
aquecimento.unlink()

# --- Escrita
t0 = time.time(); cotacoes_ok.to_parquet(caminho_parquet, index=False); t_esc_pq = time.time() - t0
t0 = time.time(); cotacoes_ok.to_csv(caminho_csv, index=False);         t_esc_csv = time.time() - t0

# --- Leitura completa
t0 = time.time(); lido_pq = pd.read_parquet(caminho_parquet);           t_lei_pq = time.time() - t0
t0 = time.time(); lido_csv = pd.read_csv(caminho_csv);                  t_lei_csv = time.time() - t0

# --- Leitura de APENAS 3 colunas (onde o formato colunar brilha)
cols = ["Date", "Ticker", "Adj Close"]
t0 = time.time(); _ = pd.read_parquet(caminho_parquet, columns=cols);   t_col_pq = time.time() - t0
t0 = time.time(); _ = pd.read_csv(caminho_csv, usecols=cols);           t_col_csv = time.time() - t0

mb = lambda p: p.stat().st_size / 1024**2

comparacao = pd.DataFrame({
    "formato":       ["Parquet", "CSV"],
    "tamanho_MB":    [mb(caminho_parquet), mb(caminho_csv)],
    "escrita_s":     [t_esc_pq, t_esc_csv],
    "leitura_s":     [t_lei_pq, t_lei_csv],
    "leitura_3col_s": [t_col_pq, t_col_csv],
}).set_index("formato")

print(comparacao.round(4).to_string())
print(f"\nEspaco em disco: Parquet e {mb(caminho_csv) / mb(caminho_parquet):.1f}x menor")

### 🔎 Leia esse resultado com honestidade

**Tamanho**: a vantagem do Parquet é grande e inquestionável — tipicamente **3x ou mais**. Colunas
de preços têm valores muito parecidos entre si, e a compressão colunar explora exatamente isso.

**Tempo**: aqui a sua base tem dezenas de milhares de linhas — é **pequena**. Nessa escala os
tempos são de centésimos de segundo e a comparação é dominada por ruído; o CSV pode até ganhar em
alguns casos. A diferença vira abismo quando a base cresce para milhões de linhas, porque o custo
do CSV é **parsear texto** e cresce linearmente com o tamanho do arquivo.

**Leitura de 3 colunas**: esta é a comparação mais reveladora. O Parquet lê **só os blocos daquelas
colunas** e ignora fisicamente o resto do arquivo. O `usecols` do CSV **descarta** colunas depois de
já ter lido e parseado o arquivo inteiro — a economia é de memória, não de trabalho.

> 🎯 A moral do benchmark não é "Parquet é sempre mais rápido". É: **meça no seu volume de dados,
> com aquecimento, antes de acreditar em qualquer regra geral** — inclusive nesta.

In [ ]:
# O ponto que costuma passar batido: o que acontece com os TIPOS
print("Tipo da coluna Date apos reler de cada formato:")
print(f"  Parquet -> {lido_pq['Date'].dtype}")
print(f"  CSV     -> {lido_csv['Date'].dtype}")
print()
print("No CSV a data voltou como TEXTO. Toda operacao temporal (diff, resample, filtro por")
print("periodo) quebra ate voce lembrar de chamar pd.to_datetime -- e essa conversao custa")
print("tempo toda vez que o arquivo e lido.")

In [ ]:
# Salvando tambem a base macro
caminho_macro = DATA_DIR / "b3_macro.parquet"
macro.to_parquet(caminho_macro, index=False)

print("Arquivos gerados em", DATA_DIR.resolve(), "\n")
for f in sorted(DATA_DIR.glob("*")):
    print(f"  {f.name:28s} {f.stat().st_size / 1024**2:8.2f} MB")

### 📌 Quando o CSV ainda faz sentido

Não é dogma. CSV continua sendo a escolha certa quando:

- alguém vai **abrir o arquivo no Excel** ou ler a olho nu;
- o dado é **pequeno** (algumas centenas de linhas) e a diferença é irrelevante;
- você precisa versionar no Git e **ver o diff** linha a linha (Parquet é binário: o diff é inútil).

Para tudo que é volumoso, gerado por código e consumido por código, **Parquet**.

---

---

# FASE 7 — COLETA INCREMENTAL

## §7.1 — O problema

Você vai rodar essa coleta **todo dia**. Baixar 10 anos de histórico toda vez é:

- **lento** — minutos que viram horas quando o universo cresce para 1.500 papéis;
- **frágil** — quanto mais requisições, maior a chance de tomar *rate limit* ou timeout;
- **abusivo** — você está consumindo um serviço gratuito para rebaixar dados que já tem.

A coleta incremental baixa **apenas o que falta**:

```
existe o arquivo?
├── NÃO → coleta completa (10 anos) e salva
└── SIM → para cada ticker:
          ├── JÁ está no arquivo  → baixa de (última data − buffer) até hoje
          └── é NOVO na lista     → baixa o histórico completo dele
          → junta tudo, remove duplicatas mantendo o mais recente, salva
```

### 🔎 Por que o "buffer" de 2 dias?

Por que não baixar a partir da última data exata? Porque a fonte **reescreve o passado**:

- proventos e *splits* são aplicados retroativamente em **toda** a série `Adj Close`;
- pregões são consolidados com atraso e valores podem ser corrigidos no dia seguinte.

Rebaixar os dois últimos dias e **sobrescrever** o que estava salvo custa quase nada e evita que
você carregue para sempre um valor desatualizado. É uma troca consciente: um pouquinho de trabalho
repetido em troca de dados corretos.

In [ ]:
def atualizar_cotacoes(caminho, tickers, buffer_dias: int = 2,
                       periodo: str = "10y") -> pd.DataFrame:
    """Coleta incremental: baixa so o que falta e devolve a base completa.

    Args:
        caminho:     arquivo .parquet onde a base vive.
        tickers:     universo desejado (pode ter tickers novos em relacao ao arquivo).
        buffer_dias: quantos dias reprocessar antes da ultima data salva.
        periodo:     janela usada quando e preciso baixar historico completo.
    """
    caminho = Path(caminho)

    # Caso 1: nao existe base -> coleta completa
    if not caminho.exists():
        print("Sem base anterior -- coleta COMPLETA.")
        df = coletar_cotacoes(tickers, periodo=periodo)
        if not df.empty:
            df.to_parquet(caminho, index=False)
        return df

    antigo = pd.read_parquet(caminho)
    ja_temos = set(antigo["Ticker"].unique())
    existentes = [t for t in tickers if t in ja_temos]
    novos = [t for t in tickers if t not in ja_temos]
    print(f"Base atual: {len(antigo):,} linhas | {len(existentes)} tickers conhecidos, "
          f"{len(novos)} novos")

    partes = [antigo]

    # Caso 2: tickers ja conhecidos -> so a ponta recente
    if existentes:
        ultima = antigo.loc[antigo["Ticker"].isin(existentes), "Date"].max()
        inicio = (ultima - pd.Timedelta(days=buffer_dias)).strftime("%Y-%m-%d")
        print(f"  incremental a partir de {inicio} (ultima data salva: {ultima.date()})")
        recente = coletar_cotacoes(existentes, start=inicio)
        if not recente.empty:
            partes.append(recente)

    # Caso 3: tickers novos -> historico completo deles
    if novos:
        print(f"  historico completo para: {novos}")
        cheio = coletar_cotacoes(novos, periodo=periodo)
        if not cheio.empty:
            partes.append(cheio)

    df = pd.concat(partes, ignore_index=True)

    # keep='last' e o coracao do metodo: como o que acabou de ser baixado entra DEPOIS
    # no concat, ele SOBRESCREVE a versao antiga da mesma (Date, Ticker).
    antes = len(df)
    df = (df.drop_duplicates(subset=["Date", "Ticker"], keep="last")
            .sort_values(["Ticker", "Date"])
            .reset_index(drop=True))

    print(f"  {antes:,} linhas apos concat -> {len(df):,} apos deduplicar "
          f"({len(df) - len(antigo):+,} vs. base anterior)")
    df.to_parquet(caminho, index=False)
    return df

In [ ]:
# Primeira chamada: o arquivo ja existe (salvamos na Fase 6), entao vai ser incremental.
print("--- Execucao 1 ---")
base = atualizar_cotacoes(caminho_parquet, ATIVOS_OK)

In [ ]:
# Segunda chamada seguida: nada mudou no mercado desde ha 10 segundos.
# O delta deve ser ZERO -- e essa e exatamente a prova de que a deduplicacao funciona.
print("--- Execucao 2 (logo em seguida) ---")
base = atualizar_cotacoes(caminho_parquet, ATIVOS_OK)

In [ ]:
# Agora com um ticker NOVO: ele deve baixar o historico completo so desse papel.
print("--- Execucao 3 (universo ampliado) ---")
UNIVERSO_AMPLIADO = ATIVOS_OK + ["MGLU3.SA"]
base = atualizar_cotacoes(caminho_parquet, UNIVERSO_AMPLIADO)

print()
print(base.groupby("Ticker")["Date"].agg(["count", "min", "max"]).tail(4))

### 📌 O que esse teste provou

1. **Execução 1** — baixou apenas a ponta recente (últimos dias), não os 10 anos.
2. **Execução 2** — delta de **0 linhas**. A deduplicação por `(Date, Ticker)` está correta: rodar
   duas vezes seguidas **não** duplica nada. Essa propriedade tem nome: a função é
   **idempotente**, e é o que permite agendar a coleta sem medo.
3. **Execução 3** — o ticker novo entrou com histórico completo; os antigos, só com a ponta.

> 💡 Numa base de 1.500 papéis, isso é a diferença entre **4 minutos** e **20 segundos** por
> execução. É o que torna viável rodar a coleta todo dia às 19h, depois do fechamento.

---

---

# FASE 8 — A BASE SERVE PARA MODELAR?

Coletamos, auditamos, guardamos. Falta a pergunta final: **esse dado responde perguntas?**

Três verificações rápidas — e nenhuma delas é decorativa: cada uma é o primeiro passo de um tipo de
modelo diferente.

## §8.1 — Desempenho comparado (base 100)

In [ ]:
# Formato LARGO: uma coluna por ativo, indexado por data. E o formato que
# matrizes de correlacao, otimizadores de carteira e modelos multivariados esperam.
precos = (base.pivot_table(index="Date", columns="Ticker", values="Adj Close")
              .sort_index())

# ffill: repete o ultimo preco conhecido para feriados de um mercado que nao sao do outro.
# NUNCA bfill -- isso traria preco do FUTURO para o passado (look-ahead bias).
precos = precos.ffill()

print("Formato largo:", precos.shape, "(pregoes x ativos)")
precos.tail(3).round(2)

In [ ]:
selecao = [t for t in ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "WEGE3.SA",
                       "KNRI11.SA", "XFIX11.SA", "^BVSP"] if t in precos.columns]

# Ultimos 5 anos, reindexado para base 100 no primeiro pregao da janela
corte = precos.index.max() - pd.Timedelta(days=5 * 365)
janela = precos.loc[precos.index >= corte, selecao].dropna(how="all")
base100 = janela / janela.iloc[0] * 100

fig, ax = plt.subplots(figsize=(13, 5))
for col in base100.columns:
    ax.plot(base100.index, base100[col], label=col, lw=1.3)
ax.axhline(100, color="gray", ls="--", lw=0.8)
ax.set(title="Quanto viraria R$ 100 investidos no inicio da janela (Adj Close)",
       xlabel="Data", ylabel="Indice (inicio = 100)")
ax.legend(ncol=3, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 🔎 Por que **base 100** e não o preço em reais

Porque preços em reais **não são comparáveis**: um ativo de R$ 8 e outro de R$ 40 no mesmo eixo
dizem apenas qual é mais caro por unidade — o que é irrelevante. Dividir tudo pelo primeiro valor e
multiplicar por 100 responde à pergunta que interessa: **quanto rendeu**.

A linha cinza tracejada em 100 é o ponto de equilíbrio. Acima: ganho. Abaixo: prejuízo.

> 📌 Repare que estamos usando `Adj Close`. Se tivéssemos usado `Close`, os FIIs apareceriam
> praticamente na horizontal — e a conclusão seria falsa, pelo motivo que vimos na Fase 3.

---
## §8.2 — Os ativos andam juntos? (correlação)

In [ ]:
# Retorno diario: e nele que a correlacao deve ser calculada, nunca no PRECO.
# Precos sao nao-estacionarios (tem tendencia) -- duas series que so sobem tem
# correlacao alta mesmo sem nenhuma relacao real entre elas (correlacao espuria).
retornos = precos[selecao].pct_change().dropna()
corr = retornos.corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(corr)), corr.index, fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlacao dos retornos diarios")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### 📌 O que a matriz revela

- **Ações entre si**: correlação alta (0,4 a 0,7). Todas respondem ao mesmo humor do mercado
  brasileiro — juros, câmbio, política.
- **Ação vs. `^BVSP`**: a mais alta de todas, e é esperado — o Ibovespa **é** uma média ponderada
  dessas mesmas ações. Não é descoberta, é definição.
- **FII vs. ação**: bem mais baixa. É justamente por isso que FII entra em carteira: ele **não cai
  junto**, e é essa a definição operacional de diversificação.

> 🎯 Correlação baixa entre classes é o que faz a diversificação funcionar. Um otimizador de
> carteira (Markowitz, HRP) consome **exatamente esta matriz** como entrada. Ou seja: a matriz que
> você acabou de calcular já é o primeiro ingrediente de uma alocação quantitativa.

---
## §8.3 — Juntando mercado e macro

In [ ]:
# A Selic e diaria e o ETF tambem -- mas os calendarios nao coincidem.
# Solucao: reindexar a macro no calendario da BOLSA e propagar para frente (ffill).
selic = (macro[macro["Indicador"] == "selic"]
         .set_index("Date")["Valor"]
         .sort_index())

# Taxa diaria (% ao dia) -> equivalente anualizada, com 252 pregoes no ano
selic_anual = ((1 + selic / 100) ** 252 - 1) * 100

ativo_fii = "XFIX11.SA" if "XFIX11.SA" in precos.columns else "KNRI11.SA"
serie_fii = precos[ativo_fii].dropna()

# Passo a passo do alinhamento:
#   1) juntar os dois calendarios,  2) propagar a Selic para frente (ffill),
#   3) reindexar no calendario da BOLSA,  4) descartar o inicio sem Selic conhecida.
selic_al = (selic_anual
            .reindex(selic_anual.index.union(serie_fii.index))
            .ffill()
            .reindex(serie_fii.index))
mask = selic_al.notna()
fii_al, selic_al = serie_fii[mask], selic_al[mask]

fig, ax1 = plt.subplots(figsize=(13, 4.5))
ax1.plot(fii_al.index, fii_al, color="tab:blue", lw=1.3, label=ativo_fii)
ax1.set_ylabel(ativo_fii, color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(selic_al.index, selic_al, color="tab:red", lw=1.3, label="Selic (% a.a.)")
ax2.set_ylabel("Selic anualizada (%)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

ax1.set(title=f"{ativo_fii} vs. Selic anualizada", xlabel="Data")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

corr_nivel = np.corrcoef(fii_al.values, selic_al.values)[0, 1]
print(f"Correlacao (em NIVEL, cuidado): {corr_nivel:+.3f}")
print(f"Correlacao (em VARIACAO mensal): "
      f"{fii_al.pct_change(21).corr(selic_al.diff(21)):+.3f}")

### ⚠️ Duas correlações, dois significados — e uma armadilha

A correlação **em nível** (preço bruto vs. taxa bruta) é a que quase todo mundo calcula primeiro, e
é a **menos confiável**. Ambas as séries têm tendência de longo prazo; duas séries com tendência
exibem correlação forte mesmo **sem nenhuma relação causal**. É a *correlação espúria* — o mesmo
mecanismo que faz o consumo de queijo "correlacionar" com mortes por lençol emaranhado.

A correlação **em variação** (quanto o FII mudou em ~1 mês vs. quanto a Selic mudou no mesmo mês)
compara **movimentos**, não níveis. É essa que tem chance de significar algo.

A teoria prevê sinal **negativo**: quando a Selic sobe, o investidor ganha mais na renda fixa sem
risco, então passa a exigir *yield* maior do FII — e, como o aluguel pago pelo fundo é relativamente
fixo, o único jeito de o *yield* subir é **a cota cair**.

Na prática, o número que você obteve é provavelmente **negativo, mas fraco** (algo entre −0,3 e 0),
e muda conforme a janela escolhida. Isso não invalida a teoria: significa que a Selic é **um** dos
fatores, não o único — expectativa de inflação, vacância e risco de crédito puxam para outros lados
ao mesmo tempo.

> 📌 Este é o tipo de variável exógena que alimenta modelos de previsão: informação que **não está**
> no histórico de preços do próprio ativo. Fraca isoladamente, útil em conjunto — é assim que a
> maioria das *features* macro se comporta.

> ⚠️ E não tire conclusão de uma correlação só: troque a janela, troque o ativo e veja se o sinal
> se mantém. Um número que só aparece numa configuração específica é ruído com aparência de achado.

---

---

# 🎓 FECHAMENTO — O QUE VOCÊ CONSTRUIU

Um coletor de dados de mercado completo e reaproveitável:

| Componente | Função |
|------------|--------|
| `listar_papeis_fundamentus()` | descobre o universo investível sem viés de memória |
| `coletar_cotacoes()` | baixa OHLCV e normaliza para formato longo |
| `coletar_macro()` / `coletar_sgs_rest()` | traz séries do BCB, com plano B se a lib falhar |
| checagens 1–5 | auditam nulos, duplicatas, volume, coerência e saltos |
| `atualizar_cotacoes()` | coleta incremental idempotente, pronta para agendar |

### As cinco ideias que sobrevivem a esta aula

1. **`Adj Close` para retorno, `Close` para execução.** Confundir os dois inverte a conclusão de
   qualquer estudo com dividendos.
2. **O universo vem de uma fonte, não da sua memória.** Senão você embute viés de sobrevivência.
3. **Toda fonte externa é suspeita até auditada** — e auditar cabe em cinco checagens.
4. **Nunca preencha para trás.** `ffill` é informação que você tinha; `bfill` é informação do futuro.
5. **Coleta incremental e idempotente.** Rodar duas vezes tem que dar o mesmo resultado que rodar
   uma vez.

### O que ainda falta nesta base

Coletamos **preço** e **macro**. Falta o dinheiro que o ativo **paga**: dividendos, JCP e
rendimentos de FII. Como vimos na Fase 3, essa parcela responde por uma fatia enorme do retorno —
em FIIs, praticamente por todo ele. É o assunto do notebook seguinte,
**`coleta_dividendos_b3.ipynb`**, que inclui uma auditoria cruzada bonita: reconstruir os proventos
a partir do próprio ajuste de preço e conferir contra a lista informada pela fonte.

### Para a próxima aula

Com a base completa, o caminho natural é: **engenharia de atributos** (médias móveis, volatilidade,
RSI, defasagens) → **modelo** (a MLP da Aula 5, agora em série temporal) → **backtest**.

E aí entra a regra que não pode ser violada: em série temporal, a divisão treino/teste é sempre
**cronológica**. Nada de `train_test_split(shuffle=True)` — embaralhar datas permite que o modelo
treine com o futuro e preveja o passado, e a métrica resultante é ficção.

---

---

# 📝 EXERCÍCIOS

### Nível 1 — aquecimento

**1.1** Amplie `UNIVERSO` com três ações de setores que ainda não aparecem (ex.: `ABEV3.SA`,
`RENT3.SA`, `SUZB3.SA`). Rode `atualizar_cotacoes()` e confirme, na saída, que **só** os três novos
baixaram histórico completo.

**1.2** Calcule, para cada ativo da base, o **retorno anualizado** dos últimos 5 anos:
$\left(\frac{P_{fim}}{P_{inicio}}\right)^{252/n} - 1$, onde $n$ é o número de pregões. Ordene do
maior para o menor.

**1.3** Acrescente a série **câmbio PTAX venda** (SGS 1) ao `SGS_CODIGOS` e recolete a macro.
Confira as unidades no resumo.

### Nível 2 — análise

**2.1** Compare, com o `Close`, o **retorno total** de `KNRI11.SA` em 10 anos contra o mesmo cálculo
com `Adj Close`. Quantos pontos percentuais de retorno "desaparecem" ao usar a coluna errada?

**2.2** Escreva `proventos_implicitos(df, ticker)` que reconstrua os proventos a partir da razão
`Adj Close / Close`: na data-ex esse fator dá um degrau, e o tamanho do degrau é o valor pago.
Compare o resultado com `yf.Ticker(ticker).dividends` — as duas fontes devem bater.
(Este exercício é resolvido e aprofundado no notebook `coleta_dividendos_b3.ipynb`.)

**2.2b** E os desdobramentos? Verifique **primeiro** se o Yahoo já aplica o split na coluna `Close`,
comparando `yf.Ticker(ticker).splits` com o preço no dia anterior e no dia do evento. A resposta
muda completamente o que um detector de split deveria procurar — e descobrir isso **é** o exercício.

**2.3** Construa a matriz de correlação dos retornos usando **só** os pregões em que **todos** os
ativos negociaram (`dropna(how="any")`) e compare com a versão que usa `ffill`. As correlações
mudam? Qual das duas você defenderia, e por quê?

### Nível 3 — engenharia

**3.1** Implemente `coletar_dividendos(tickers)` usando `yf.Ticker(tk).dividends`. Salve em
`data/raw/b3_dividendos.parquet` no formato longo `(Date, Ticker, Valor)`.

**3.2** Adapte `atualizar_cotacoes()` para também ser incremental na base **macro**, deduplicando
por `(Date, Indicador)`.

**3.3** Transforme as funções desta aula em um módulo `coleta_b3.py` com uma função `main()` que
atualize tudo. Rode pelo terminal: `python coleta_b3.py`.

### Nível 4 — desafio

**4.1** A coleta de 1.500 tickers de uma vez costuma falhar. Implemente **coleta em lotes** de 50
tickers com `time.sleep(1)` entre lotes e **retry com espera exponencial** (1s, 2s, 4s) nos lotes
que falharem. Meça quanto tempo leva o universo completo.

**4.2** Escreva `validar_base(df)` que rode as cinco checagens da Fase 5 e devolva um dicionário
`{"nulos": ..., "duplicatas": ..., "aprovado": True/False}`, levantando `ValueError` se houver
duplicata na chave primária. É assim que se transforma uma checagem manual em **teste automático**
de pipeline.

**4.3** Pesquise o arquivo **COTAHIST** da B3 (fonte oficial, layout de largura fixa) e escreva um
leitor com `pd.read_fwf`. Compare o `Close` oficial da B3 com o do Yahoo para `PETR4` em 2024: eles
batem? Onde divergem, e por quê?

---